<a href="https://colab.research.google.com/github/Michaeldw012/Retail-Dashboard/blob/main/RETAIL_DASHBOARD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# CREATE RETAIL DASHBOARD FILE FOR STREAMLIT
# Run this cell once to create the file
# ============================================

dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import random

# Page configuration
st.set_page_config(
    page_title="Retail Analytics Dashboard",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================
# GENERATE SAMPLE DATA
# ============================================

@st.cache_data
def generate_data():
    """Generate 2 years of retail data"""
    np.random.seed(42)

    dates = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
    stores = ['Store A', 'Store B', 'Store C', 'Store D', 'Store E']
    categories = ['Electronics', 'Clothing', 'Home', 'Beauty', 'Toys', 'Food', 'Books']

    data = []
    for date in dates:
        for store in stores:
            for category in categories:
                # Base sales with patterns
                weekday = date.weekday()
                is_weekend = weekday >= 5
                month = date.month

                # Seasonal patterns
                if month in [6, 7, 8]:
                    season_factor = 1.3
                elif month == 12:
                    season_factor = 1.5
                else:
                    season_factor = 1.0

                # Weekend boost
                weekend_factor = 1.4 if is_weekend else 1.0

                # Base sales
                base_sales = random.randint(50, 200)
                sales = int(base_sales * season_factor * weekend_factor * random.uniform(0.8, 1.2))

                # Inventory
                inventory = random.randint(sales, sales * 3)

                # Customer count
                customers = int(sales * random.uniform(0.3, 0.7))

                data.append({
                    'date': date,
                    'store': store,
                    'category': category,
                    'sales': sales,
                    'inventory': inventory,
                    'customers': customers,
                    'revenue': sales * random.uniform(20, 100)
                })

    return pd.DataFrame(data)

df = generate_data()

# ============================================
# SIDEBAR FILTERS
# ============================================

st.sidebar.title("🎛️ Filters")

# Date range
min_date = df['date'].min().date()
max_date = df['date'].max().date()
date_range = st.sidebar.date_input(
    "Select Date Range",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date
)

# Store filter
stores = ['All'] + sorted(df['store'].unique().tolist())
selected_store = st.sidebar.selectbox("Select Store", stores)

# Category filter
categories = ['All'] + sorted(df['category'].unique().tolist())
selected_category = st.sidebar.selectbox("Select Category", categories)

# Apply filters
filtered_df = df.copy()
if len(date_range) == 2:
    start_date, end_date = date_range
    filtered_df = filtered_df[
        (filtered_df['date'].dt.date >= start_date) &
        (filtered_df['date'].dt.date <= end_date)
    ]
if selected_store != 'All':
    filtered_df = filtered_df[filtered_df['store'] == selected_store]
if selected_category != 'All':
    filtered_df = filtered_df[filtered_df['category'] == selected_category]

# ============================================
# MAIN DASHBOARD
# ============================================

# Title
st.title("📊 Retail Analytics Dashboard")
st.markdown(f"*Showing data from {filtered_df['date'].min().strftime('%B %d, %Y')} to {filtered_df['date'].max().strftime('%B %d, %Y')}*")
st.markdown("---")

# ============================================
# KPI CARDS
# ============================================

col1, col2, col3, col4 = st.columns(4)

with col1:
    total_revenue = filtered_df['revenue'].sum()
    st.metric(
        label="💰 Total Revenue",
        value=f"${total_revenue:,.0f}",
        delta=f"{((total_revenue / df['revenue'].sum()) * 100):.1f}% of total"
    )

with col2:
    total_sales = filtered_df['sales'].sum()
    st.metric(
        label="📦 Total Sales (Units)",
        value=f"{total_sales:,.0f}",
        delta=f"{filtered_df['store'].nunique()} stores"
    )

with col3:
    avg_sales_per_day = filtered_df.groupby('date')['sales'].sum().mean()
    st.metric(
        label="📈 Avg Daily Sales",
        value=f"{avg_sales_per_day:,.0f} units",
        delta=f"{filtered_df['date'].nunique()} days"
    )

with col4:
    total_customers = filtered_df['customers'].sum()
    st.metric(
        label="👥 Total Customers",
        value=f"{total_customers:,.0f}",
        delta=f"{(total_customers / df['customers'].sum() * 100):.1f}%"
    )

st.markdown("---")

# ============================================
# ROW 1: SALES TREND + CATEGORY BREAKDOWN
# ============================================

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("📈 Sales Trend")

    # Daily sales trend
    daily_sales = filtered_df.groupby('date')['sales'].sum().reset_index()

    fig_trend = px.line(
        daily_sales,
        x='date',
        y='sales',
        title='Daily Sales Volume',
        labels={'date': 'Date', 'sales': 'Sales (Units)'},
        line_shape='spline'
    )
    fig_trend.update_layout(height=400)
    st.plotly_chart(fig_trend, use_container_width=True)

with col2:
    st.subheader("🏷️ Sales by Category")

    # Category breakdown
    category_sales = filtered_df.groupby('category')['sales'].sum().reset_index()
    category_sales = category_sales.sort_values('sales', ascending=True)

    fig_category = px.bar(
        category_sales,
        x='sales',
        y='category',
        title='Sales by Category',
        labels={'sales': 'Sales (Units)', 'category': ''},
        color='sales',
        color_continuous_scale='Blues',
        orientation='h'
    )
    fig_category.update_layout(height=400)
    st.plotly_chart(fig_category, use_container_width=True)

# ============================================
# ROW 2: STORE PERFORMANCE + REVENUE ANALYSIS
# ============================================

col1, col2 = st.columns(2)

with col1:
    st.subheader("🏪 Store Performance")

    store_performance = filtered_df.groupby('store').agg({
        'sales': 'sum',
        'revenue': 'sum',
        'customers': 'sum'
    }).reset_index()

    fig_store = px.bar(
        store_performance,
        x='store',
        y='revenue',
        title='Revenue by Store',
        labels={'store': 'Store', 'revenue': 'Revenue ($)'},
        color='revenue',
        color_continuous_scale='Greens',
        text_auto='.2s'
    )
    fig_store.update_layout(height=350)
    st.plotly_chart(fig_store, use_container_width=True)

with col2:
    st.subheader("📊 Revenue vs Sales")

    # Daily revenue vs sales
    daily_metrics = filtered_df.groupby('date').agg({
        'sales': 'sum',
        'revenue': 'sum'
    }).reset_index()

    fig_dual = go.Figure()
    fig_dual.add_trace(go.Bar(
        x=daily_metrics['date'],
        y=daily_metrics['sales'],
        name='Sales (Units)',
        marker_color='lightblue',
        yaxis='y'
    ))
    fig_dual.add_trace(go.Scatter(
        x=daily_metrics['date'],
        y=daily_metrics['revenue'],
        name='Revenue ($)',
        marker_color='darkgreen',
        yaxis='y2',
        line=dict(width=3)
    ))
    fig_dual.update_layout(
        title='Sales vs Revenue',
        height=350,
        yaxis=dict(title='Sales (Units)'),
        yaxis2=dict(title='Revenue ($)', overlaying='y', side='right')
    )
    st.plotly_chart(fig_dual, use_container_width=True)

st.markdown("---")

# ============================================
# ROW 3: INSIGHTS & RECOMMENDATIONS
# ============================================

st.subheader("💡 Business Insights")

col1, col2, col3 = st.columns(3)

with col1:
    # Top performing category
    top_category = filtered_df.groupby('category')['revenue'].sum().idxmax()
    top_revenue = filtered_df.groupby('category')['revenue'].sum().max()

    st.info(
        f"**🏆 Top Category**\n\n"
        f"**{top_category}** leads with **${top_revenue:,.0f}** in revenue.\n\n"
        f"Consider increasing inventory and marketing for this category."
    )

with col2:
    # Best store
    best_store = filtered_df.groupby('store')['revenue'].sum().idxmax()
    best_revenue = filtered_df.groupby('store')['revenue'].sum().max()

    st.success(
        f"**🏪 Best Store**\n\n"
        f"**{best_store}** generates **${best_revenue:,.0f}** in revenue.\n\n"
        f"Analyze what makes this store successful and replicate."
    )

with col3:
    # Recommendation
    total_sales_all = filtered_df['sales'].sum()
    avg_sales = filtered_df['sales'].mean()

    st.warning(
        f"**📈 Opportunity**\n\n"
        f"Current average daily sales: **{avg_sales:.0f}** units.\n\n"
        f"Increasing sales by 10% could generate **${total_sales_all * 0.1 * 50:,.0f}** in additional revenue."
    )

# ============================================
# FOOTER
# ============================================

st.markdown("---")
st.markdown(
    """
    <div style='text-align: center; color: gray;'>
        <p>📊 Retail Analytics Dashboard | Built with Streamlit</p>
    </div>
    """,
    unsafe_allow_html=True
)
'''

# Write the file
with open('retail_dashboard.py', 'w') as f:
    f.write(dashboard_code)

print("✅ retail_dashboard.py created successfully!")
print("\n📋 NEXT STEPS:")
print("1. Click 'File' → 'Save a copy in GitHub'")
print("2. Repository: Michaeldw012/Retail-Dashboard")
print("3. File path: retail_dashboard.py")
print("4. Click OK")
print("\n5. Create requirements.txt file (see below)")
print("6. Go to share.streamlit.io and deploy!")

✅ retail_dashboard.py created successfully!

📋 NEXT STEPS:
1. Click 'File' → 'Save a copy in GitHub'
2. Repository: Michaeldw012/Retail-Dashboard
3. File path: retail_dashboard.py
4. Click OK

5. Create requirements.txt file (see below)
6. Go to share.streamlit.io and deploy!


In [2]:
# Create requirements.txt file
requirements_content = """
streamlit
pandas
numpy
plotly
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements_content.strip())

print("✅ requirements.txt created successfully!")
print("📋 Now you have two files:")
print("   - retail_dashboard.py")
print("   - requirements.txt")

✅ requirements.txt created successfully!
📋 Now you have two files:
   - retail_dashboard.py
   - requirements.txt
